# kaggle-vllm 0.1.2 — Focused Runtime-Reset Acceptance

This notebook validates only the filesystem-reset behavior added in `kaggle-vllm==0.1.2`, followed by a short real TP=2 inference smoke test. It intentionally does **not** download the multi-GB Qwen checkpoint.

Required Kaggle settings:

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Python: the documented Kaggle CPython 3.12 runtime

The notebook uses only its owned runtime paths below. It never broadly deletes `/kaggle/working`, never prints secrets, and expects the download cache to survive reset. Creating this notebook is not acceptance evidence; only a fully executed copy ending in `FINAL ACCEPTANCE: PASS` is evidence.

## 1. Verify the Kaggle dual-T4 environment

In [1]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

EXPECTED_NATIVE_REPO = "waqasm86/kaggle-vllm-binaries"
EXPECTED_NATIVE_REVISION = "f6b4f10de54924ed6fe9e28cceab84eca7276ab6"
EXPECTED_NATIVE_WHEEL = "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl"
EXPECTED_NATIVE_SHA256 = "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c"
EXPECTED_TORCH = "2.10.0+cu128"
EXPECTED_TORCH_CUDA = "12.8"

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi", "-L"], check=True)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(
        f"GPU {index}:",
        torch.cuda.get_device_name(index),
        "capability",
        torch.cuda.get_device_capability(index),
    )
print("NCCL:", ".".join(map(str, torch.cuda.nccl.version())))

assert sys.version_info[:2] == (3, 12), sys.version
assert torch.__version__ == EXPECTED_TORCH, torch.__version__
assert torch.version.cuda == EXPECTED_TORCH_CUDA, torch.version.cuda
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2
assert all("Tesla T4" in torch.cuda.get_device_name(i) for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))

TORCH_BEFORE = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
acceptance = {"environment_dual_t4": True}

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
GPU 0: Tesla T4 (UUID: GPU-5e990dd8-5cf8-ee43-0176-34a94e0760c5)
GPU 1: Tesla T4 (UUID: GPU-b2172c66-8d52-7033-ff0e-2fa7e6b226ba)
Torch: 2.10.0+cu128
Torch CUDA: 12.8
CUDA available: True
GPU count: 2
GPU 0: Tesla T4 capability (7, 5)
GPU 1: Tesla T4 capability (7, 5)
NCCL: 2.27.5


## 2. Install the exact lightweight SDK from public PyPI

This installs Hub support but does not install vLLM, Torch, or CUDA as normal dependencies.

In [2]:
%pip install --no-cache-dir --upgrade "kaggle-vllm[hub]==0.1.2"

Note: you may need to restart the kernel to use updated packages.


In [3]:
import kaggle_vllm

print("kaggle_vllm version:", kaggle_vllm.__version__)
assert kaggle_vllm.__version__ == "0.1.2"

CLI = shutil.which("kaggle-vllm")
assert CLI, "kaggle-vllm console script was not installed"
print("CLI:", CLI)

def run_cli(arguments, *, check=True):
    result = subprocess.run(
        [CLI, *arguments],
        text=True,
        capture_output=True,
        check=False,
    )
    print("$", CLI, *arguments)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    print("return code:", result.returncode)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with return code {result.returncode}")
    return result

def parse_cli_json(result):
    # A real bootstrap runs pip subprocesses before the CLI emits its final JSON.
    # Parse the last top-level JSON object while preserving all preceding logs.
    starts = [index for index, char in enumerate(result.stdout) if char == "{"]
    for index in reversed(starts):
        try:
            return json.loads(result.stdout[index:])
        except json.JSONDecodeError:
            continue
    raise AssertionError("CLI output did not end with a JSON object")

run_cli(["fingerprint"])
run_cli(["verify-gpus", "--tensor-parallel-size", "2"])
acceptance["pypi_0_1_2"] = True

kaggle_vllm version: 0.1.2
CLI: /usr/local/bin/kaggle-vllm
$ /usr/local/bin/kaggle-vllm fingerprint
{
  "is_kaggle": true,
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "torch_path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py",
  "torch_cuda": "12.8",
  "cuda_available": true,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    }
  ],
  "nccl": "2.27.5",
  "nvcc": "/usr/local/cuda/bin/nvcc",
  "nvcc_version": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "cuda_home": "/usr/local/cuda",
  "cuda_driver": "/

## 3. Select isolated notebook-owned runtime paths

A fresh session must start without staged/overlay/manifest state at this root. The reusable wheel cache is outside the reset root and may already exist. The notebook refuses to clean unexpected prior state automatically.

In [5]:
RUNTIME_ROOT = Path("/kaggle/working/kaggle-vllm-e2e-012")
STAGED = RUNTIME_ROOT / "vllm-staged"
OVERLAY = RUNTIME_ROOT / "vllm-runtime-overlay"
MANIFEST = RUNTIME_ROOT / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")
REFUSAL_PROBE_MANIFEST = RUNTIME_ROOT / "non-empty-refusal-probe.json"

for path in (STAGED, OVERLAY, MANIFEST, REFUSAL_PROBE_MANIFEST):
    assert not path.exists(), (
        f"Fresh acceptance path required; {path} already exists. "
        "Start a fresh Kaggle session or choose a new notebook-owned RUNTIME_ROOT."
    )

print("staged:", STAGED)
print("overlay:", OVERLAY)
print("manifest:", MANIFEST)
print("preserved cache:", CACHE)

BOOTSTRAP_BASE = [
    "bootstrap",
    "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(MANIFEST),
]

staged: /kaggle/working/kaggle-vllm-e2e-012/vllm-staged
overlay: /kaggle/working/kaggle-vllm-e2e-012/vllm-runtime-overlay
manifest: /kaggle/working/kaggle-vllm-e2e-012/kaggle-vllm-runtime.json
preserved cache: /kaggle/working/kaggle-vllm-cache


## 4. Strict dry-run and immutable artifact identity

In [6]:
dry_result = run_cli([*BOOTSTRAP_BASE, "--dry-run", "--json"])
dry_data = parse_cli_json(dry_result)
assert dry_data["profile"] == "kaggle-t4x2-cu128"
assert dry_data["compatible"] is True
assert dry_data["artifact"]["hf_repo_id"] == EXPECTED_NATIVE_REPO
assert dry_data["artifact"]["hf_revision"] == EXPECTED_NATIVE_REVISION
assert dry_data["artifact"]["filename"] == EXPECTED_NATIVE_WHEEL
assert dry_data["artifact"]["sha256"] == EXPECTED_NATIVE_SHA256
assert not any(path.exists() for path in (STAGED, OVERLAY, MANIFEST))
acceptance["strict_dry_run"] = True

$ /usr/local/bin/kaggle-vllm bootstrap --strict --staged /kaggle/working/kaggle-vllm-e2e-012/vllm-staged --overlay /kaggle/working/kaggle-vllm-e2e-012/vllm-runtime-overlay --cache /kaggle/working/kaggle-vllm-cache --manifest /kaggle/working/kaggle-vllm-e2e-012/kaggle-vllm-runtime.json --dry-run --json
{
  "profile": "kaggle-t4x2-cu128",
  "strict": true,
  "compatible": true,
  "findings": [
    {
      "check": "Python implementation",
      "status": "pass",
      "message": "Python implementation: CPython"
    },
    {
      "check": "Python ABI",
      "status": "pass",
      "message": "Python ABI: cp312"
    },
    {
      "check": "operating system",
      "status": "pass",
      "message": "operating system: Linux"
    },
    {
      "check": "machine",
      "status": "pass",
      "message": "machine: x86_64"
    },
    {
      "check": "Kaggle runtime",
      "status": "pass",
      "message": "Kaggle runtime: True"
    },
    {
      "check": "PyTorch",
      "status": "pas

## 5. First strict bootstrap

This downloads or reuses the immutable wheel, verifies SHA256, stages the native runtime and locked overlay, and writes the ownership manifest.

In [7]:
initial_bootstrap = run_cli(BOOTSTRAP_BASE)
assert MANIFEST.is_file()
assert STAGED.is_dir() and any(STAGED.iterdir())
assert OVERLAY.is_dir() and any(OVERLAY.iterdir())
assert CACHE.is_dir()
manifest_data = json.loads(MANIFEST.read_text(encoding="utf-8"))
assert manifest_data["wheel"]["hf_repo_id"] == EXPECTED_NATIVE_REPO
assert manifest_data["wheel"]["hf_revision"] == EXPECTED_NATIVE_REVISION
assert manifest_data["wheel"]["sha256"] == EXPECTED_NATIVE_SHA256
acceptance["initial_strict_bootstrap"] = True

$ /usr/local/bin/kaggle-vllm bootstrap --strict --staged /kaggle/working/kaggle-vllm-e2e-012/vllm-staged --overlay /kaggle/working/kaggle-vllm-e2e-012/vllm-runtime-overlay --cache /kaggle/working/kaggle-vllm-cache --manifest /kaggle/working/kaggle-vllm-e2e-012/kaggle-vllm-runtime.json
Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 262.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 326.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 376.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 336.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 270.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/7

## 6. Prove the default non-empty-destination refusal

An exact matching completed manifest is intentionally reusable/idempotent. To exercise the original non-empty safety guard without modifying the completed runtime, this probe keeps the same non-empty staged, overlay, and cache paths but selects an absent manifest. It must fail before any overwrite.

In [8]:
REFUSAL_ARGS = [
    "bootstrap",
    "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(REFUSAL_PROBE_MANIFEST),
]
refusal = run_cli(REFUSAL_ARGS, check=False)
refusal_output = refusal.stdout + refusal.stderr
assert refusal.returncode != 0
assert "refusing to overwrite non-empty" in refusal_output
assert STAGED.is_dir() and OVERLAY.is_dir() and MANIFEST.is_file()
assert not REFUSAL_PROBE_MANIFEST.exists()
acceptance["expected_non_empty_refusal"] = True
print("PASS: normal bootstrap safely refused an unowned non-empty destination.")

$ /usr/local/bin/kaggle-vllm bootstrap --strict --staged /kaggle/working/kaggle-vllm-e2e-012/vllm-staged --overlay /kaggle/working/kaggle-vllm-e2e-012/vllm-runtime-overlay --cache /kaggle/working/kaggle-vllm-cache --manifest /kaggle/working/kaggle-vllm-e2e-012/non-empty-refusal-probe.json
kaggle-vllm: error: refusing to overwrite non-empty staged wheel destination: /kaggle/working/kaggle-vllm-e2e-012/vllm-staged
return code: 2
PASS: normal bootstrap safely refused an unowned non-empty destination.


## 7. Reset dry-run must be non-mutating

In [9]:
def path_snapshot(path):
    if not path.exists():
        return {"exists": False}
    if path.is_file():
        stat = path.lstat()
        return {"exists": True, "kind": "file", "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
    entries = []
    for entry in sorted(path.rglob("*")):
        stat = entry.lstat()
        entries.append((str(entry.relative_to(path)), entry.is_symlink(), stat.st_size, stat.st_mtime_ns))
    return {"exists": True, "kind": "directory", "entries": entries}

before_dry_run = {
    "staged": path_snapshot(STAGED),
    "overlay": path_snapshot(OVERLAY),
    "manifest": path_snapshot(MANIFEST),
    "cache": path_snapshot(CACHE),
}
cache_before_reset = before_dry_run["cache"]

reset_dry = run_cli([*BOOTSTRAP_BASE, "--reset-runtime", "--dry-run", "--json"])
reset_dry_data = parse_cli_json(reset_dry)
assert reset_dry_data["reset"]["safe"] is True
assert reset_dry_data["reset"]["completed"] is False
actions = {target["label"]: target["action"] for target in reset_dry_data["reset"]["targets"]}
assert actions == {"staged": "remove", "overlay": "remove", "manifest": "remove", "cache": "preserve"}

after_dry_run = {
    "staged": path_snapshot(STAGED),
    "overlay": path_snapshot(OVERLAY),
    "manifest": path_snapshot(MANIFEST),
    "cache": path_snapshot(CACHE),
}
assert after_dry_run == before_dry_run
acceptance["reset_dry_run_non_mutating"] = True
print("PASS: reset dry-run changed no selected filesystem state.")

$ /usr/local/bin/kaggle-vllm bootstrap --strict --staged /kaggle/working/kaggle-vllm-e2e-012/vllm-staged --overlay /kaggle/working/kaggle-vllm-e2e-012/vllm-runtime-overlay --cache /kaggle/working/kaggle-vllm-cache --manifest /kaggle/working/kaggle-vllm-e2e-012/kaggle-vllm-runtime.json --reset-runtime --dry-run --json
{
  "profile": "kaggle-t4x2-cu128",
  "strict": true,
  "compatible": true,
  "findings": [
    {
      "check": "Python implementation",
      "status": "pass",
      "message": "Python implementation: CPython"
    },
    {
      "check": "Python ABI",
      "status": "pass",
      "message": "Python ABI: cp312"
    },
    {
      "check": "operating system",
      "status": "pass",
      "message": "operating system: Linux"
    },
    {
      "check": "machine",
      "status": "pass",
      "message": "machine: x86_64"
    },
    {
      "check": "Kaggle runtime",
      "status": "pass",
      "message": "Kaggle runtime: True"
    },
    {
      "check": "PyTorch",
    

## 8. Explicit reset, cache preservation, and immediate strict re-bootstrap

In [10]:
confirmed = run_cli([*BOOTSTRAP_BASE, "--reset-runtime", "--yes", "--json"])
confirmed_data = parse_cli_json(confirmed)
assert confirmed_data["reset"]["completed"] is True
assert confirmed_data["completed"] is True
assert confirmed_data["already_complete"] is False
assert STAGED.is_dir() and any(STAGED.iterdir())
assert OVERLAY.is_dir() and any(OVERLAY.iterdir())
assert MANIFEST.is_file()
assert CACHE.is_dir()
assert path_snapshot(CACHE) == cache_before_reset
recreated_manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
assert recreated_manifest["wheel"]["hf_revision"] == EXPECTED_NATIVE_REVISION
assert recreated_manifest["wheel"]["sha256"] == EXPECTED_NATIVE_SHA256
acceptance["explicit_reset"] = True
acceptance["cache_preserved"] = True
acceptance["strict_rebootstrap"] = True
print("PASS: owned runtime reset, cache preserved, strict bootstrap recreated runtime.")

$ /usr/local/bin/kaggle-vllm bootstrap --strict --staged /kaggle/working/kaggle-vllm-e2e-012/vllm-staged --overlay /kaggle/working/kaggle-vllm-e2e-012/vllm-runtime-overlay --cache /kaggle/working/kaggle-vllm-cache --manifest /kaggle/working/kaggle-vllm-e2e-012/kaggle-vllm-runtime.json --reset-runtime --yes --json
Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 23.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 197.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 326.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 328.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 395.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 282.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

## 9. Activate the recreated runtime and verify native extensions

In [11]:
os.environ["KAGGLE_VLLM_MANIFEST"] = str(MANIFEST)
from kaggle_vllm.bootstrap import activate_runtime

assert activate_runtime(MANIFEST)

import vllm
import vllm._C
import vllm._moe_C
import vllm.cumem_allocator

native_modules = {
    "vllm": vllm,
    "vllm._C": vllm._C,
    "vllm._moe_C": vllm._moe_C,
    "vllm.cumem_allocator": vllm.cumem_allocator,
}
for name, module in native_modules.items():
    module_path = Path(module.__file__).resolve()
    print(name, module_path)
    assert module_path.is_relative_to(STAGED.resolve()), (name, module_path)
print("vLLM version:", vllm.__version__)
acceptance["native_imports"] = True

vllm /kaggle/working/kaggle-vllm-e2e-012/vllm-staged/vllm/__init__.py
vllm._C /kaggle/working/kaggle-vllm-e2e-012/vllm-staged/vllm/_C.abi3.so
vllm._moe_C /kaggle/working/kaggle-vllm-e2e-012/vllm-staged/vllm/_moe_C.abi3.so
vllm.cumem_allocator /kaggle/working/kaggle-vllm-e2e-012/vllm-staged/vllm/cumem_allocator.abi3.so
vLLM version: 0.18.2.dev0+ga26e8dc7f.d20260822


## 10. Confirm Kaggle Torch was preserved

In [12]:
TORCH_AFTER = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
print("Torch before:", TORCH_BEFORE)
print("Torch after:", TORCH_AFTER)
assert TORCH_AFTER == TORCH_BEFORE
assert torch.cuda.device_count() == 2
acceptance["torch_preserved"] = True

Torch before: {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py'}
Torch after: {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py'}


## 11. Focused real TP=2 OPT-125M inference

The engine logs should show world size 2, NCCL, and TP ranks 0 and 1. FlashAttention 2 unavailability and SymmMem capability warnings are expected on SM75; the validated attention fallback is `TRITON_ATTN`.

In [13]:
from kaggle_vllm import KaggleLLM
from vllm import SamplingParams

llm = KaggleLLM(
    model="facebook/opt-125m",
    tensor_parallel_size=2,
    max_model_len=512,
    gpu_memory_utilization=0.60,
)
sampling = SamplingParams(temperature=0.0, max_tokens=32)
outputs = llm.generate(
    ["Kaggle dual NVIDIA T4 runtime reset acceptance is"],
    sampling,
)
assert outputs and outputs[0].outputs and outputs[0].outputs[0].text
print("PROMPT:", outputs[0].prompt)
print("OUTPUT:", outputs[0].outputs[0].text)
assert llm.tensor_parallel_size == 2
acceptance["opt_tp2_generation"] = True

INFO 08-25 04:02:00 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 512, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': 'facebook/opt-125m'}


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

INFO 08-25 04:02:21 [model.py:533] Resolved architecture: OPTForCausalLM
INFO 08-25 04:02:21 [model.py:1582] Using max model len 512
INFO 08-25 04:02:22 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-25 04:02:22 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 08-25 04:02:22 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-25 04:02:22 [vllm.py:820] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 08-25 04:02:22 [vllm.py:985] Cudagraph is disabled under eager mode
INFO 08-25 04:02:22 [compilation.py:289] Enabled custom fusions: norm_quant, act_quant


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.99it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.99it/s]
(Worker_TP0 pid=329) 


(Worker_TP0 pid=329) INFO 08-25 04:03:03 [default_loader.py:384] Loading weights took 0.26 seconds
(Worker_TP0 pid=329) INFO 08-25 04:03:04 [gpu_model_runner.py:4566] Model loading took 0.12 GiB memory and 4.407234 seconds
(Worker_TP0 pid=329) INFO 08-25 04:03:23 [gpu_worker.py:456] Available KV cache memory: 8.39 GiB
(EngineCore pid=305) INFO 08-25 04:03:23 [kv_cache_utils.py:1316] GPU KV cache size: 488,736 tokens
(EngineCore pid=305) INFO 08-25 04:03:23 [kv_cache_utils.py:1321] Maximum concurrency for 512 tokens per request: 954.56x
(EngineCore pid=305) INFO 08-25 04:03:24 [core.py:281] init engine (profile, create kv cache, warmup model) took 20.36 seconds
(EngineCore pid=305) INFO 08-25 04:03:26 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=305) WARNING 08-25 04:03:26 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=305) WARNING 08-25 04:03:26 [vllm.py:82

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

PROMPT: Kaggle dual NVIDIA T4 runtime reset acceptance is
OUTPUT:  enabled.

The Kaggle dual NVIDIA T4 runtime reset acceptance is enabled.

The Kaggle dual NVIDIA T4 runtime reset acceptance is


## 12. Final focused acceptance summary

In [14]:
required_checks = [
    "environment_dual_t4",
    "pypi_0_1_2",
    "strict_dry_run",
    "initial_strict_bootstrap",
    "expected_non_empty_refusal",
    "reset_dry_run_non_mutating",
    "explicit_reset",
    "cache_preserved",
    "strict_rebootstrap",
    "native_imports",
    "torch_preserved",
    "opt_tp2_generation",
]
print(json.dumps(acceptance, indent=2, sort_keys=True))
missing = [name for name in required_checks if acceptance.get(name) is not True]
if missing:
    print("FINAL ACCEPTANCE: FAIL")
    raise AssertionError(f"acceptance checks missing or false: {missing}")
print("FINAL ACCEPTANCE: PASS")

{
  "cache_preserved": true,
  "environment_dual_t4": true,
  "expected_non_empty_refusal": true,
  "explicit_reset": true,
  "initial_strict_bootstrap": true,
  "native_imports": true,
  "opt_tp2_generation": true,
  "pypi_0_1_2": true,
  "reset_dry_run_non_mutating": true,
  "strict_dry_run": true,
  "strict_rebootstrap": true,
  "torch_preserved": true
}
FINAL ACCEPTANCE: PASS
